[데이터셋 정보]
- 출처: NIH Chest X-ray 14 Dataset
- 다운로드: https://www.kaggle.com/datasets/nih-chest-xrays/data
- 규모: 112,120장 (30,000+ 환자), 14개 질환 레이블

### Chapter 5.2: Chest X-ray 폐렴 판독 AI
#
#### DenseNet121을 활용한 흉부 X-ray 다중 질환 분류

##### 1. 라이브러리 설치 및 임포트

In [ ]:
# 필수 라이브러리 설치
# !pip install torch torchvision kaggle scikit-learn pillow pandas numpy matplotlib

In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

##### 2. 데이터셋 다운로드

Kaggle API 설정 (최초 1회)
1. https://www.kaggle.com/settings/account 접속
2. 'Create New API Token' 클릭
3. kaggle.json을 ~/.kaggle/ 폴더에 저장

In [ ]:
# 데이터 디렉토리 설정
DATA_DIR = './data/chest_xray'
os.makedirs(DATA_DIR, exist_ok=True)

print(f"데이터 저장 경로: {DATA_DIR}")

In [ ]:
# Kaggle API로 데이터 다운로드 (수동 다운로드 가능)
# 주석 해제하여 실행:
# !kaggle datasets download -d nih-chest-xrays/data -p {DATA_DIR}

# 또는 리사이즈 버전 (용량 절약):
# !kaggle datasets download -d khanfashee/nih-chest-x-ray-14-224x224-resized -p {DATA_DIR}

print("💡 수동 다운로드:")
print("https://www.kaggle.com/datasets/nih-chest-xrays/data")
print(f"다운로드 후 {DATA_DIR} 폴더에 압축 해제")

##### 3. 데이터 전처리

In [ ]:
# ImageNet 표준 정규화 값
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# 학습용 transform (데이터 증강)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize
])

# 평가용 transform
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

print("✅ Transform 정의 완료")

##### 4. 데이터셋 클래스 정의

In [ ]:
# 14개 흉부 질환 클래스
CLASS_NAMES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]

print(f"질환 클래스 ({len(CLASS_NAMES)}개):")
for i, name in enumerate(CLASS_NAMES, 1):
    print(f"  {i}. {name}")

In [ ]:
# CSV 파일 로드 (데이터셋에 포함)
CSV_FILE = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')

# 실제 파일 존재 확인
if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
    print(f"✅ CSV 로드 완료: {len(df)}개 샘플")
    print(f"\n데이터 샘플:")
    print(df.head())
else:
    print(f"⚠️  CSV 파일이 없습니다: {CSV_FILE}")
    print("데이터 다운로드 후 다시 실행하세요.")
    df = None

In [ ]:
# 레이블 인코딩 (multi-hot vector)
if df is not None:
    labels = []

    for idx, row in df.iterrows():
        finding = row['Finding Labels']
        label = np.zeros(len(CLASS_NAMES), dtype=np.float32)

        if finding != 'No Finding':
            diseases = finding.split('|')
            for disease in diseases:
                if disease in CLASS_NAMES:
                    label[CLASS_NAMES.index(disease)] = 1.0

        labels.append(label)

    labels = np.array(labels)
    print(f"✅ 레이블 인코딩 완료: {labels.shape}")
    print(f"샘플 레이블: {labels[0]}")

##### 5. 데이터 로더 생성


In [ ]:
# 하이퍼파라미터
BATCH_SIZE = 32
NUM_WORKERS = 4

# 간단한 Dataset 클래스 (인라인)
class ChestXrayDataset(Dataset):
    def __init__(self, image_dir, dataframe, labels, transform=None):
        self.image_dir = Path(image_dir)
        self.df = dataframe
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['Image Index']
        img_path = self.image_dir / 'images' / img_name

        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        label = torch.FloatTensor(self.labels[idx])
        return image, label

In [ ]:
# 데이터셋 생성
if df is not None:
    IMAGE_DIR = DATA_DIR

    dataset = ChestXrayDataset(
        image_dir=IMAGE_DIR,
        dataframe=df,
        labels=labels,
        transform=train_transform
    )

    # 데이터 로더
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    print(f"✅ 데이터 로더 생성 완료")
    print(f"  - 총 샘플: {len(dataset)}")
    print(f"  - 배치 크기: {BATCH_SIZE}")
    print(f"  - 배치 수: {len(dataloader)}")

##### 6. DenseNet121 모델 생성

In [ ]:
# DenseNet121 모델 로드
model = models.densenet121(pretrained=True)

# 마지막 classifier 교체
num_ftrs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 14),  # 14개 클래스
    nn.Sigmoid()  # Multi-label을 위한 sigmoid
)

print("✅ DenseNet121 모델 생성 완료")
print(f"  - 입력 feature: {num_ftrs}")
print(f"  - 출력 클래스: 14")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"🖥️  디바이스: {device}")

##### 7. 손실 함수 및 최적화

In [ ]:
# Binary Cross Entropy Loss (multi-label)
criterion = nn.BCELoss()

# Adam optimizer
LEARNING_RATE = 0.001
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"✅ 손실 함수: BCELoss")
print(f"✅ 최적화: Adam (lr={LEARNING_RATE})")

##### 8. 학습 루프

In [ ]:
NUM_EPOCHS = 5  # 테스트용 (실제로는 50+ 권장)

print(f"학습 시작: {NUM_EPOCHS} 에포크")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0

    for batch_idx, (images, labels_batch) in enumerate(dataloader):
        # GPU로 이동
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        # 순전파
        optimizer.zero_grad()
        outputs = model(images)

        # 손실 계산
        loss = criterion(outputs, labels_batch)

        # 역전파
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # 진행 상황 출력
        if (batch_idx + 1) % 10 == 0:
            print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}], "
                  f"Batch [{batch_idx+1}/{len(dataloader)}], "
                  f"Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(dataloader)
    print(f"✅ Epoch {epoch+1} 완료 - 평균 Loss: {avg_loss:.4f}")
    print("-"*60)

print("학습 완료")

##### 9. 모델 평가 (AUROC)

In [ ]:
model.eval()

all_labels = []
all_predictions = []

print("모델 평가 중...")

with torch.no_grad():
    for images, labels_batch in dataloader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        # 예측
        outputs = model(images)

        # CPU로 이동
        all_labels.append(labels_batch.cpu().numpy())
        all_predictions.append(outputs.cpu().numpy())

# 배치 결합
all_labels = np.vstack(all_labels)
all_predictions = np.vstack(all_predictions)

print(f"예측 완료: {all_labels.shape}")

# 각 클래스별 AUROC 계산
aurocs = []

for i in range(len(CLASS_NAMES)):
    if all_labels[:, i].sum() > 0:  # 양성 샘플이 있는 경우만
        auroc = roc_auc_score(all_labels[:, i], all_predictions[:, i])
        aurocs.append(auroc)
    else:
        aurocs.append(np.nan)

# 결과 출력
print("\n" + "="*70)
print("📊 클래스별 AUROC 결과")
print("="*70)

for i, (name, auroc) in enumerate(zip(CLASS_NAMES, aurocs)):
    if not np.isnan(auroc):
        print(f"{i+1:2d}. {name:25s}: {auroc:.4f}")
    else:
        print(f"{i+1:2d}. {name:25s}: N/A")

# 평균 AUROC
valid_aurocs = [a for a in aurocs if not np.isnan(a)]
if valid_aurocs:
    avg_auroc = np.mean(valid_aurocs)
    print("="*70)
    print(f"✅ 평균 AUROC: {avg_auroc:.4f}")
    print("="*70)

##### 10. 모델 저장

In [ ]:
# 모델 저장
SAVE_DIR = './saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

model_path = os.path.join(SAVE_DIR, 'densenet121_chest_xray.pth')
torch.save(model.state_dict(), model_path)

print(f"✅ 모델 저장 완료: {model_path}")

##### 11. 단일 이미지 예측 (추론)

In [ ]:
# 테스트 이미지 경로 설정 (실제 이미지로 변경 필요)
test_image_path = os.path.join(DATA_DIR, 'images', '00000001_000.png')

# 이미지가 존재하는 경우 예측
if os.path.exists(test_image_path):
    # 이미지 로드 및 전처리
    test_image = Image.open(test_image_path).convert('RGB')
    test_tensor = val_transform(test_image).unsqueeze(0).to(device)

    # 예측
    model.eval()
    with torch.no_grad():
        output = model(test_tensor)
        probabilities = output.cpu().numpy()[0]

    # 결과 출력
    print("\n📋 예측 결과:")
    print("="*50)

    for name, prob in zip(CLASS_NAMES, probabilities):
        print(f"{name:25s}: {prob:.4f} ({prob*100:.1f}%)")

    # 상위 3개 질환
    top3_indices = np.argsort(probabilities)[-3:][::-1]
    print("\n🏆 상위 3개 질환:")
    for idx in top3_indices:
        print(f"  - {CLASS_NAMES[idx]}: {probabilities[idx]:.4f}")

    # 이미지 시각화
    plt.figure(figsize=(8, 8))
    plt.imshow(test_image, cmap='gray')
    plt.title('Test Chest X-ray')
    plt.axis('off')
    plt.show()
else:
    print(f"⚠️  테스트 이미지를 찾을 수 없음: {test_image_path}")